> 在线讲义：[第 25 章 · 编译与部署：PTX、cubin、fatbin、JIT，以及 Rust host + C++ kernel](https://wikieden.github.io/cuda-to-tensor-core/lessons/25-compile-deploy.html) · 本书《CUDA 从线程到 Tensor Core》

# 第 25 章 · 编译与部署：PTX、cubin、fatbin、JIT，以及 Rust host + C++ kernel

本章回答：`nvcc` 把一个 `.cu` 文件变成了什么？PTX 和 SASS 有什么区别？为什么一份程序能在好几代 GPU 上跑、有时又跑不了？运行时编译（NVRTC）有什么用？生产里 Rust 程序怎么调 C++ 写的 kernel？自己写的 kernel 怎么发布给别人用？

配套讲义：仓库里 `cuda-programming/notes/lessons/25-compile-deploy.md`（讲解、动画、练习都在讲义里）。对应原文：PG §1.3.3 PTX、§1.3.4 Cubins and Fatbins（含 Binary / PTX Compatibility、JIT，`materials/text/cuda-programming-guide.txt` 第 1428–1545 行）、§2.7 NVCC（第 7592 行起，PTX 与 cubin 生成在 §2.7.3.1 第 7707 行）；03 Rust vs C++ §5 选型决策；01 Rust 生态；04 调研 §3.4（Hugging Face Kernels Hub）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU：代码用 NumPy 在 CPU 上模拟 GPU 的规则（线程编号、32 字节一箱、32 个 bank……）。

In [ ]:
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第三步 规则 1 | cubin：同一大版本、小版本不低于它 | `can_run("sm_80", cc)` | 8.0、8.6、8.9 可以，9.0 不行 |
| 第三步 规则 2 | PTX 向前兼容 | `can_run("compute_80", cc)` | 7.5 不行；8.0、9.0、10.0 都 JIT |
| 第三步 规则 3 | 带 a 的只认自己 | `can_run("sm_90a", cc)` | 9.0 可以，10.0 不行 |
| 第二步 fatbin | 驱动挑最合适的一份 | `pick_from_fatbin(["sm_80", "sm_90a", "compute_90"], cc)` | 8.6 → sm_80；9.0 → sm_90a；10.0 → compute_90 JIT；7.5 → 跑不了 |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `parse`**（来自 `ch25_compile_deploy.py`）：'sm_86' → ('sm', 8, 6, False)；'compute_90a' → ('compute', 9, 0, True)。

In [ ]:
def parse(target: str) -> tuple[str, int, int, bool]:
    """'sm_86' → ('sm', 8, 6, False)；'compute_90a' → ('compute', 9, 0, True)。"""
    kind, ver = target.split("_")
    arch_specific = ver.endswith("a")
    ver = ver.rstrip("a")
    return kind, int(ver[:-1]), int(ver[-1]), arch_specific

**函数 `can_run`**（来自 `ch25_compile_deploy.py`）：返回 'cubin'（直接跑）、'JIT'（驱动把 PTX 现场编译）或 None（跑不了）。

In [ ]:
def can_run(target: str, cc: tuple[int, int]) -> str | None:
    """返回 'cubin'（直接跑）、'JIT'（驱动把 PTX 现场编译）或 None（跑不了）。"""
    kind, major, minor, a = parse(target)
    if a:                                              # sm_90a / sm_100a：只认同一个架构
        return ("cubin" if kind == "sm" else "JIT") if (major, minor) == cc else None
    if kind == "sm":                                   # cubin：同一大版本、小版本不低于它
        return "cubin" if cc[0] == major and cc[1] >= minor else None
    return "JIT" if cc >= (major, minor) else None

**函数 `nvcc_flags`**（来自 `ch25_compile_deploy.py`）：-gencode arch=compute_XX,code=sm_XX 的命令行。

In [ ]:
def nvcc_flags(pairs: list[tuple[str, str]]) -> str:
    """-gencode arch=compute_XX,code=sm_XX 的命令行。"""
    return " ".join(f"-gencode arch={a},code={c}" for a, c in pairs)

**函数 `pick_from_fatbin`**（来自 `ch25_compile_deploy.py`）：驱动从 fatbin 里挑：能直接跑的 cubin 优先（选版本最高的），没有再用 PTX JIT。

In [ ]:
def pick_from_fatbin(targets: list[str], cc: tuple[int, int]) -> tuple[str, str] | None:
    """驱动从 fatbin 里挑：能直接跑的 cubin 优先（选版本最高的），没有再用 PTX JIT。"""
    ok = [(t, can_run(t, cc)) for t in targets]
    ok = [(t, how) for t, how in ok if how]
    if not ok:
        return None
    ok.sort(key=lambda th: (th[1] == "cubin", parse(th[0])[1:3]), reverse=True)
    return ok[0]

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第三步 规则 1：cubin 只往同一大版本的更高小版本兼容（以 sm_80 为例）")
for cc in ((8, 0), (8, 6), (8, 9), (9, 0)):
    print(f"  sm_80 cubin 在 CC {cc[0]}.{cc[1]} 上：{can_run('sm_80', cc)}")

In [ ]:
print("\n第三步 规则 2：PTX 能向前兼容（JIT）")
for cc in ((7, 5), (8, 0), (9, 0), (10, 0)):
    print(f"  compute_80 PTX 在 CC {cc[0]}.{cc[1]} 上：{can_run('compute_80', cc)}")

In [ ]:
print("\n第三步 规则 3：带 a 的只认自己")
for cc in ((9, 0), (10, 0)):
    print(f"  sm_90a 在 CC {cc[0]}.{cc[1]} 上：{can_run('sm_90a', cc)}")

In [ ]:
print("\n第二步 讲义里那条 nvcc 命令装进 fatbin 的三份")
pairs = [("compute_80", "sm_80"), ("compute_90a", "sm_90a"), ("compute_90", "compute_90")]
print("  nvcc", nvcc_flags(pairs), "my.cu")
fatbin = [c for _, c in pairs]
for cc in ((8, 0), (8, 6), (9, 0), (10, 0), (7, 5)):
    print(f"  CC {cc[0]}.{cc[1]:<2} 的卡挑：{pick_from_fatbin(fatbin, cc)}")

## 3. 代码里才看得到的

1. **Python 的元组比较是逐项比**：`(10, 0) >= (8, 0)` 为 True，正好就是"CC 不低于"的意思。
2. **fatbin 里 cubin 优先于 PTX**：能直接跑的机器码不用现场编译；都不能直接跑时才 JIT（首次加载慢，结果会缓存）。
3. **讲义那条 nvcc 命令只给 `sm_90a` 编了 cubin、没带 `compute_90a` 的 PTX**：所以 10.0 的卡用的是普通 `compute_90` 那份 PTX——里面不能有 wgmma 这类 9.0a 专属指令。

## 4. 自己改一改

- 往 fatbin 里加一份 `sm_100a`，看 10.0 的卡改挑哪一份。
- 把 fatbin 换成只有一份 `compute_80` PTX，看 8.0、9.0、10.0 的卡各挑什么、哪些要 JIT。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> PTX 是菜谱、SASS 是操作卡，ptxas 和驱动负责改写；
> nvcc 一分为二：host 给 C++ 编译器，device 变 PTX 再变 cubin，装进 fatbin；
> cubin 只认同代更高小版本，PTX 能给未来的卡现场编；带 a 的专属指令不往后兼容；
> 运行时编译：参数现场定、宿主不是 C++ 时用；
> 生产默认 C++ kernel + Rust host；发 kernel 交给 Kernels Hub 按版本组合编。